# BackgroundFX - Google Colab Migration with Google Drive

Welcome! This notebook runs your BackgroundFX app in Colab and saves processed videos permanently to your Google Drive.

## How to Use:

1.  **Run `Step 1`** to install necessary software.
2.  **Run `Step 2`** to connect to your Google Drive. You will need to authorize access.
3.  **Run `Step 3`** to write the application code to a file.
4.  **Get a free `ngrok` authtoken** from [your ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
5.  **Run `Step 4`**. Paste your `ngrok` authtoken when prompted.
6.  **Click the public URL** that appears. This opens your app. Processed videos will be saved to a `BackgroundFX_Output` folder in your Google Drive.

---_

In [ ]:
# Step 1: Install Dependencies
!pip install streamlit opencv-python-headless numpy Pillow requests pyngrok -q

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Write the Streamlit App Code to a File
%%writefile app.py

import streamlit as st
import cv2
import numpy as np
import tempfile
import os
from PIL import Image
import requests
from io import BytesIO
import logging
import shutil

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Google Drive Save Path ---
GDRIVE_OUTPUT_DIR = '/content/drive/MyDrive/BackgroundFX_Output'
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

def load_background_image(background_url):
    try:
        response = requests.get(background_url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
        return np.array(image.convert('RGB'))
    except Exception as e:
        logger.error(f'Failed to load background image: {e}')
        return create_default_background()

def create_default_background():
    height, width = 720, 1280
    background = np.ones((height, width, 3), dtype=np.uint8) * 150
    brick_height, brick_width = 40, 80
    for y in range(0, height, brick_height):
        for x in range(0, width, brick_width):
            offset = brick_width // 2 if (y // brick_height) % 2 else 0
            x_pos = (x + offset) % width
            cv2.rectangle(background, (x_pos, y), (min(x_pos + brick_width - 2, width), min(y + brick_height - 2, height)), (180, 120, 80), -1)
            cv2.rectangle(background, (x_pos, y), (min(x_pos + brick_width - 2, width), min(y + brick_height - 2, height)), (120, 80, 40), 2)
    return background

def get_professional_backgrounds():
    return {
        "Brick Wall": "https://images.unsplash.com/photo-1558618666-fcd25c85cd64?w=1280&h=720&fit=crop",
        "Simple Office": "https://images.unsplash.com/photo-1497366216548-37526070297c?w=1280&h=720&fit=crop",
        "Executive Office": "https://images.unsplash.com/photo-1586953208448-b95a79798f07?w=1280&h=720&fit=crop",
        "Modern Conference Room": "https://images.unsplash.com/photo-1560472354-b33ff0c44a43?w=1280&h=720&fit=crop"
    }

def segment_person_fallback(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_RGB2HSV)
    lower_skin = np.array([0, 20, 70])
    upper_skin = np.array([20, 255, 255])
    skin_mask = cv2.inRange(hsv, lower_skin, upper_skin)
    kernel = np.ones((5, 5), np.uint8)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(skin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        mask = np.zeros(frame.shape[:2], dtype=np.uint8)
        cv2.fillPoly(mask, [largest_contour], 255)
        kernel = np.ones((20, 20), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=2)
        return mask.astype(bool)
    return None

def insert_green_screen(frame, person_mask):
    green_background = np.zeros_like(frame)
    green_background[:, :] = [0, 255, 0]
    return np.where(person_mask[..., None], frame, green_background)

def chroma_key_replacement(green_screen_frame, new_background):
    h, w = green_screen_frame.shape[:2]
    background_resized = cv2.resize(new_background, (w, h))
    hsv = cv2.cvtColor(green_screen_frame, cv2.COLOR_RGB2HSV)
    lower_green = np.array([40, 50, 50])
    upper_green = np.array([80, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    mask_normalized = green_mask.astype(float) / 255
    result = green_screen_frame.copy()
    for c in range(3):
        result[:, :, c] = (green_screen_frame[:, :, c] * (1 - mask_normalized) + background_resized[:, :, c] * mask_normalized)
    return result.astype(np.uint8)

def process_video(video_path, background_image, progress_callback=None):
    try:
        cap = cv2.VideoCapture(video_path)
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        temp_output_path = tempfile.mktemp(suffix='.mp4')
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(temp_output_path, fourcc, fps, (width, height))
        frame_count = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            person_mask = segment_person_fallback(frame_rgb)
            if person_mask is not None:
                green_screen_frame = insert_green_screen(frame_rgb, person_mask)
                final_frame = chroma_key_replacement(green_screen_frame, background_image)
            else:
                final_frame = frame_rgb
            out.write(cv2.cvtColor(final_frame, cv2.COLOR_RGB2BGR))
            frame_count += 1
            if progress_callback: progress_callback(frame_count / total_frames, f'Processing frame {frame_count}/{total_frames}')
        cap.release()
        out.release()
        # Save to Google Drive
        drive_path = os.path.join(GDRIVE_OUTPUT_DIR, os.path.basename(video_path))
        shutil.copyfile(temp_output_path, drive_path)
        return temp_output_path, drive_path
    except Exception as e:
        logger.error(f'Video processing failed: {e}')
        return None, None

def main():
    st.set_page_config(page_title="BackgroundFX", page_icon="🎬", layout="wide")
    st.title("🎬 BackgroundFX - Video Background Replacement")
    if 'video_path' not in st.session_state: st.session_state.video_path = None
    col1, col2 = st.columns(2)
    with col1:
        st.markdown("### 📹 Upload Video")
        uploaded_video = st.file_uploader("Choose a video file", type=['mp4', 'mov'])
        if uploaded_video:
            with tempfile.NamedTemporaryFile(delete=False, suffix=os.path.splitext(uploaded_video.name)[1]) as tmp_file:
                tmp_file.write(uploaded_video.getvalue())
                st.session_state.video_path = tmp_file.name
            st.video(uploaded_video)
    with col2:
        st.markdown("### 🖼️ Background Image")
        background_options = get_professional_backgrounds()
        selected_background = st.selectbox("Choose background", options=list(background_options.keys()))
        background_url = background_options[selected_background]
        background_image = load_background_image(background_url)
        st.image(background_image, caption=f'Background: {selected_background}')
    if st.session_state.video_path and st.button("🎬 Process Video", type="primary"):
        with st.spinner('Processing...'):
            progress_bar = st.progress(0)
            status_text = st.empty()
            temp_path, drive_path = process_video(st.session_state.video_path, background_image, lambda p, m: (progress_bar.progress(p), status_text.text(m)))
            if temp_path:
                st.success(f'✅ Video processing complete!')
                st.info(f'💾 Saved to Google Drive: {drive_path}')
                with open(temp_path, 'rb') as f:
                    st.download_button('📥 Download Processed Video', f, file_name=os.path.basename(drive_path))
            else:
                st.error('❌ Video processing failed.')

if __name__ == "__main__":
    main()


In [ ]:
# Step 4: Run the App with ngrok
from pyngrok import ngrok

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
authtoken = input("Enter your ngrok authtoken: ")
ngrok.set_auth_token(authtoken)

# Start streamlit in the background and expose it with ngrok
public_url = ngrok.connect(8501)
print(f'

🚀 Click here to open your app: {public_url}')
!streamlit run app.py --server.port 8501 --server.headless true
